In [1]:
suppressPackageStartupMessages(library(SingleCellExperiment))
suppressPackageStartupMessages(library(scater))
suppressPackageStartupMessages(library(scran))
suppressPackageStartupMessages(library(argparse))
suppressPackageStartupMessages(library(miloR))
suppressPackageStartupMessages(library(patchwork))

In [7]:
#####################
## Define settings ##
#####################
here::i_am("processing/1_create_seurat_rna.R")
source(here::here("settings.R"))
source(here::here("utils.R"))
source(here::here("mapping/run/mnn/mapping_functions.R"))
test = TRUE

if(test){
## START TEST ##
    args = list()
args$sce <- io$rna.sce
args$metadata <- io$metadata
args$stage <- c('E7.5')#, 'E8.5', 'E9.5')
args$tdTom_corr = 'False'
args$metadata <- paste0(io$basedir,"/results/rna/mapping/sample_metadata_after_mapping.txt.gz")
args$atlas_metadata <- "/rds/project/rds-SDzz0CATGms/users/bt392/atlasses/extended/sample_metadata.txt.gz"
args$features <- 2500
args$npcs <- 50 
args$n_neighbors = 45 
args$prop = 0.15 # put at 0.15 - 0.2
args$regression = TRUE
args$remove_ExE_cells <- FALSE
args$outdir <- paste0(io$basedir,"/results/rna/MiloR/test")
## END TEST ##
}

# If passing multiple timepoints split in vector
args$stage = strsplit(args$stage, "_")[[1]] 

dir.create(args$outdir, recursive=TRUE, showWarnings = FALSE)


here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/code



In [15]:

##########################
## Load sample metadata ##
##########################

sample_metadata <- fread(args$metadata) %>%
   .[pass_rnaQC==TRUE & doublet_call==FALSE & stage %in% args$stage] %>%
    .[,pool:=stringr::str_replace_all(sample,opts$sample2pool)]


if(args$tdTom_corr!='True'){
    sample_metadata = sample_metadata[tdTom==tdTom_corr]
}

if(test){
    sample_metadata = sample_metadata[sample(1:nrow(sample_metadata), nrow(sample_metadata)/10)]
}

###############
## Load data ##
###############

# Load RNA expression data as SingleCellExperiment object
sce <- load_SingleCellExperiment(args$sce, cells=sample_metadata$cell, normalise = TRUE)

# Add sample metadata as colData
colData(sce) <- sample_metadata %>% tibble::column_to_rownames("cell") %>% DataFrame


###############
## Get PCA   ##
###############

# Get Reduced Dims
# PCA
if(length(args$stage)>1){
    message('loading precomputed PCA')
    pca_file = sprintf("%s/results/rna/dimensionality_reduction/sce/pca_features%d_pcs%d.txt.gz", io$basedir, args$features, args$npcs)
    pca = fread(pca_file)[match(colnames(sce), cell)] %>% tibble::column_to_rownames("cell") %>% as.matrix
} else {
    ## Find HVGs - detection on WT samples only
    # Get gene metadata
    gene_metadata <- fread(io$gene_metadata) %>% .[,c("chr","ens_id","symbol")] %>%
      .[symbol!="" & ens_id%in%rownames(sce)] %>%
      .[!duplicated(symbol)]

    # Imprinted genes
    imprint = gene_metadata[c(grep('maternally', gene_metadata$description),
                           grep('paternally', gene_metadata$description)), symbol]
    #Other imprinted genes: 
    #- Nnat (https://www.genecards.org/cgi-bin/carddisp.pl?gene=NNAT)
    #- Grb10 (https://www.genecards.org/cgi-bin/carddisp.pl?gene=GRB10)

    decomp <- modelGeneVar(sce[,sample_metadata[tdTom==FALSE, cell]], block=colData(sce[,sample_metadata[tdTom==FALSE, cell]])$sample) # Only detect HVGs from WT samples
    decomp <- decomp[decomp$mean > 0.01,]
    hvgs <- decomp[order(decomp$FDR),] %>% 
        as.data.table(., keep.rownames=T) %>% 
        .[grep("^Rik|Rik$|^mt-|^Rps|^Rpl|^Gm",rn,invert=T)] %>% # filter out non-informative genes
        .[grep("^Hbb|^Hba",rn,invert=T)] %>% # test removing Haem genes 
        .[!rn %in% c(imprint, 'Grb10', 'Nnat')] %>%  # remove imprinted genes
        .[!rn %in% c("Xist", "Tsix")] %>%  # remove Xist & Tsix
        .[!rn=="tomato-td"] %>% # remove tomato itself
        .[!rn%in%gene_metadata[chr=="chrY",symbol]] %>%
         head(n=args$features) %>% .$rn

    sce_filt <- sce[hvgs,]
    
    # Regress Variables
    if(args$regression){
       args$vars_to_regress = c('nFeature_RNA', 'nCount_RNA') # , 'mitochondrial_percent_RNA', 'ribosomal_percent_RNA'
       print(sprintf("Regressing out variables: %s", paste(args$vars_to_regress,collapse=" ")))
       logcounts(sce_filt) <- RegressOutMatrix(
         mtx = logcounts(sce_filt), 
         covariates = colData(sce_filt)[,args$vars_to_regress,drop=F]
        )
    }
    
    # Run PCA
    sce_filt <- runPCA(sce_filt, ncomponents = args$npcs, ntop=args$features)  
    
    # Save PCA coordinates
    outfile <- sprintf("%s/%s_pca_features%d_pcs%d.txt.gz",args$outdir, paste(args$stage, collapse ='_'), args$features, args$npcs)
    pca.dt <- reducedDim(sce_filt,"PCA") %>% round(3) %>% as.data.table(keep.rownames = T) %>% setnames("rn","cell")
    fwrite(pca.dt, outfile)
}

Warning message in .local(x, row.names, optional, ...):
“Arguments in '...' ignored”


[1] "Regressing out variables: nFeature_RNA nCount_RNA"


In [13]:
    outfile <- sprintf("%s/%s_pca_features%d_pcs%d.txt.gz",args$outdir, paste(args$stage, collapse ='_'), args$features, args$npcs)


In [14]:
outfile

[1] "/rds/project/rds-SDzz0CATGms/users/bt392/03_Stat3_RNA/results/rna/MiloR/test/E7.5_pca_features2500_pcs50.txt.gz"

In [ ]:
if(args$tdTom_corr!='True'){
    sample_metadata = sample_metadata[tdTom==tdTom_corr]
}